# Bakaano-Hydro Beginner Quickstart

This is the official beginner notebook for the core Bakaano workflow.

Use this notebook if your goal is to:

1. prepare inputs
2. compute runoff and routing
3. train a model
4. run one evaluation workflow
5. run one simulation workflow

For advanced workflows, use the dedicated notebooks for scenarios, flood mapping, or the advanced Colab/local notebooks.

If `runoff_output/rainfall_sparse_arrays.pkl` is available for the same date range, Bakaano automatically adds routed rainfall as an extra temporal predictor during training and simulation.


## Official Path

Use this notebook in this order:

1. define `working_dir` and `study_area`
2. initialize `ProjectContext`
3. inspect `workflow_overview()`, `project_paths()`, and `project_status()`
4. use `validate_project(...)` before training, evaluation, or simulation
5. run the core preprocessing, runoff/routing, training, evaluation, and simulation workflow


##### 1. Setup working directory and path to study area


In [ ]:
working_dir='/path/to/working_dir'
study_area='/path/to/study_area.shp'


##### 1b. Initialize Bakaano-Hydro and inspect project readiness


In [ ]:
from bakaano.core.project import ProjectContext

project = ProjectContext(
    working_dir=working_dir,
    study_area=study_area,
    climate_data_source='ERA5'
)

project.workflow_overview()


In [ ]:
project.project_paths()


In [ ]:
status = project.project_status()
status


In [ ]:
# Run these checks before expensive steps.
# They will raise a clear error if required preprocessing outputs are missing.
# project.validate_project(for_task='train')
# project.validate_project(for_task='evaluate')
# project.validate_project(for_task='simulate')


##### 2. Download and preprocess input data

Dataset availability notes:
- Tree cover (MODIS VCF): 2001 onward.
- NDVI (MODIS 16-day): 2001 onward.
- AlphaEarth embeddings: 2017 onward.


In [ ]:
# Get elevation data (run DEM first; other rasters align to this grid)

from bakaano.data.dem import DEM
dd = DEM(
    working_dir=working_dir, 
    study_area=study_area, 
    local_data=False, 
    local_data_path=None
)
dd.get_dem_data()
dd.plot_dem()


In [ ]:
# download and preprocess MODIS vegetation continuous fields from Google Earth Engine Data catalog

from bakaano.data.tree_cover import TreeCover
vf = TreeCover(
    working_dir=working_dir, 
    study_area=study_area, 
    start_date='2001-01-01', 
    end_date='2020-12-31'
)
vf.get_tree_cover_data()
vf.plot_tree_cover(variable='tree_cover') # options for plot are 'tree_cover' and 'herb_cover'


In [ ]:
# download and preprocess MODIS NDVI data from Google Earth Engine Data catalog

from bakaano.data.ndvi import NDVI
nd = NDVI(
    working_dir=working_dir, 
    study_area=study_area, 
    start_date='2001-01-01', 
    end_date='2010-12-31'
)
nd.get_ndvi_data()
nd.plot_ndvi(interval_num=10)  # because NDVI is in 16-day interval the 'interval_num' represents a 16-day period. 
                               #Hence 0 is the first 16 day period


In [ ]:
# Get soil data

from bakaano.data.soil import Soil
sgd = Soil(
    working_dir=working_dir, 
    study_area=study_area
)
sgd.get_soil_data()
sgd.plot_soil(variable='wilting_point')  #options are 'wilting_point', 'saturation_point' and 'available_water_content'


In [ ]:
#  Get alpha earth satellite embedding dataset

from bakaano.data.alpha_earth import AlphaEarth
dd = AlphaEarth(
    working_dir=working_dir, 
    study_area=study_area,
    start_date='2013-01-01', 
    end_date = '2024-01-01'
)
dd.get_alpha_earth()
dd.plot_alpha_earth('A35') #Band options are A00 to A63


In [ ]:
#get meteo

from bakaano.data.meteo import Meteo
cd = Meteo(
    working_dir=working_dir, 
    study_area=study_area, 
    start_date='2001-01-01', 
    end_date='2010-12-31',
    local_data=False, 
    data_source='ERA5'
)
cd.plot_meteo(variable='tasmin', date='2006-12-01') # variable options are 'tmean', 'precip', 'tasmax', 'tasmin'


##### 3. Compute runoff and route flow to the river network


In [ ]:

from bakaano.hydrology.veget import VegET
vg = VegET(
    working_dir=working_dir, 
    study_area=study_area,
    start_date='2001-01-01', 
    end_date='2010-12-31',
    climate_data_source='ERA5',
    routing_method='mfd'
)
vg.compute_veget_runoff_route_flow(resume=True, checkpoint_days=30)


In [ ]:
#visualize routed runoff data

from bakaano.hydrology.plot_runoff import RoutedRunoff
rr = RoutedRunoff(
    working_dir=working_dir, 
    study_area=study_area
)
#rr.map_routed_runoff(date='2020-01-03', vmax=7) #output values have been log transformed for better visualization


In [ ]:
rr.interactive_plot_routed_runoff_timeseries(start_date='2000-01-01', end_date='2000-12-31', 
                                             grdc_netcdf='/path/to/GRDC.nc'
                                 )


##### 4. Explore inputs, river networks, and stations interactively


In [ ]:
# Click a station marker to plot its available observed streamflow data.
# The map overlays the study area and observed station points.
rr.interactive_station_map(
    grdc_netcdf='/path/to/GRDC.nc'
)


##### 5. Train, evaluate, and apply Bakaano-Hydro 


In [ ]:
# TRAINING BAKAANO-HYDRO MODEL

# The model is trained using the GRDC streamflow data.
# Note: The training process is computationally expensive and may take a long time to complete.
# trained model is always in the models folder in the working_dir and with a .keras extension

from bakaano.neuralnet.train import asym_laplace_nll, train_streamflow_model

train_streamflow_model(
    working_dir=working_dir,
    study_area=study_area,
    train_start='1981-01-01', 
    train_end='2012-12-31', 
    grdc_netcdf='/path/to/GRDC.nc', 
    batch_size=16, 
    num_epochs=100,
    learning_rate=0.0001,
    loss_function='msle',
    model_overwrite=True,
    area_normalize=True,
    log_transform=True
)


#####   5c. Optional: Evaluate/Simulate with observed CSV files
Use the same lookup table and station CSV directory as training.


In [ ]:
# Evaluate interactively using CSV observations (enter station id when prompted)
from bakaano.neuralnet.simulate import evaluate_streamflow_model_interactively, simulate_grdc_csv_stations

model_path = f'{working_dir}/models/bakaano_model.keras'

evaluate_streamflow_model_interactively(
    working_dir=working_dir,
    study_area=study_area,
    model_path=model_path,
    val_start='2001-01-01',
    val_end='2010-12-31',
    grdc_netcdf=None,
    csv_dir='/path/to/observed_csvs',
    lookup_csv='/path/to/station_lookup.csv'
)

# Batch prediction for stations listed in lookup CSV
simulate_grdc_csv_stations(
    working_dir=working_dir,
    study_area=study_area,
    model_path=model_path,
    sim_start='1981-01-01',
    sim_end='2020-12-31',
    grdc_netcdf=None,
    csv_dir='/path/to/observed_csvs',
    lookup_csv='/path/to/station_lookup.csv'
)


In [ ]:
# EVALUATING THE TRAINED MODEL INTERACTIVELY

from bakaano.neuralnet.simulate import evaluate_streamflow_model_interactively

model_path = f'{working_dir}/models/bakaano_model.keras' 

evaluate_streamflow_model_interactively(
    working_dir=working_dir,
    study_area=study_area,
    model_path=model_path, 
    val_start='2013-01-01', 
    val_end='2020-12-31', 
    grdc_netcdf='/path/to/GRDC.nc',
)


In [ ]:
# Batch prediction for GRDC stations

from bakaano.neuralnet.simulate import simulate_grdc_csv_stations

model_path = f'{working_dir}/models/bakaano_model.keras'

simulate_grdc_csv_stations(
    working_dir=working_dir,
    study_area=study_area,
    model_path=model_path, 
    sim_start='1981-01-01', 
    sim_end='2020-12-31', 
    grdc_netcdf='/path/to/GRDC.nc'
)


In [ ]:
# PREDICTING STREAMFLOW USING THE TRAINED MODEL AND STORING AS CSV FILES 
# The model is used to predict streamflow in any location in the study area. 

from bakaano.neuralnet.simulate import simulate_streamflow

model_path = f'{working_dir}/models/bakaano_model.keras'

simulate_streamflow(
    working_dir=working_dir,
    study_area=study_area,
    model_path=model_path, 
    sim_start='1981-01-01', 
    sim_end='1990-12-31', 
    latlist=[13.8, 13.9],
    lonlist=[3.0, 4.0]
)
